# 고객·계좌·거래 데이터 무결성 검증

이 노트북은 분석을 시작하기 전에 원본 데이터의 구조와 품질을 확인합니다. 기대하는 업무 흐름은 **고객 가입일 ≤ 계좌 개설일 ≤ 거래일시**이며, 데이터 관계는 **고객 1명 ─ N개 계좌 ─ N건 거래**입니다.

핵심 날짜 규칙이 대량으로 위반되면, 기간·거래 흐름을 활용한 금융 분석 결과는 신뢰할 수 없습니다.

## 1. 데이터 불러오기

세 원본 파일을 불러옵니다. 파일명과 `DATA_DIR`은 실제 저장 위치에 맞게 수정합니다.

In [2]:
from google.colab import drive
from pathlib import Path
import pandas as pd

# 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 알려주신 데이터 경로 지정
DATA_DIR = Path('/content/drive/MyDrive/KB-Bridge/Python/Module02/미니 프로젝트/data')

customers = pd.read_csv(DATA_DIR / 'kaggle_customers.csv', encoding='utf-8-sig')
accounts = pd.read_csv(DATA_DIR / 'kaggle_accounts.csv', encoding='utf-8-sig')
transactions = pd.read_csv(DATA_DIR / 'kaggle_transactions_dirty.csv', encoding='utf-8-sig')

# 병합 키의 공백은 조인 오류를 만들 수 있으므로 먼저 제거합니다.
for frame, key in [(customers, '고객번호'), (accounts, '고객번호'),
                   (accounts, '계좌번호'), (transactions, '계좌번호'),
                   (transactions, '거래번호')]:
    frame.columns = frame.columns.str.strip()
    frame[key] = frame[key].astype('string').str.strip()

for name, frame in {'customers': customers, 'accounts': accounts, 'transactions': transactions}.items():
    print(f'{name}: {frame.shape[0]:,}행 × {frame.shape[1]}열')
    display(frame.head(3))


Mounted at /content/drive
customers: 300행 × 4열


,고객번호,도시,신용점수,가입일
0,CUS000MKX5RHTAP,South Christopherton,827,2025-12-30 00:22:11
1,CUS002V4AVJO5UQ,North Michaelport,510,2019-09-13 07:46:29
2,CUS004THQ8NDQW3,Port Faithstad,636,2024-01-01 18:57:58


accounts: 455행 × 5열


,계좌번호,고객번호,계좌유형,잔액_USD,개설일
0,ACC013JO4RI59GZ,CUS000MKX5RHTAP,Savings,151677.32,2024-06-24 05:32:41
1,ACC06TYIGXXIO7P,CUS04XM9H1NY7C3,Business,176381.95,2021-10-10 02:09:50
2,ACC080IJFJ1IACY,CUS0668LILC5RJ9,Business,175353.14,2019-10-13 17:06:01


transactions: 305행 × 4열


,거래번호,계좌번호,거래금액_USD,거래일시
0,TXNEENMT4D7JLSTY4,ACCH1YWWW9TT0Q1,"$2,252.75",2019-01-01 20:00:47
1,TXN2A0Y7NP74BA27O,ACC9L0YFOK3RHVE,"$8,951.99",2019-01-02 07:34:07
2,TXNR4NBZYEE61YFUU,ACCNOQA1TLUOOJK,"$8,071.15",2019-01-02 09:37:39


## 2. 원본 데이터 품질 점검

행 수, 결측치, 완전 중복 행, 핵심 식별자의 중복을 점검합니다. 특히 거래번호·계좌번호·고객번호는 각각의 데이터에서 식별자로 사용되므로 중복 여부를 확인해야 합니다.

In [3]:
def quality_report(name, frame, id_col):
    report = pd.DataFrame({
        '결측치 수': frame.isna().sum(),
        '결측치 비율(%)': (frame.isna().mean() * 100).round(1),
        '고유값 수': frame.nunique(dropna=True)
    })
    print(f'[{name}] 완전 중복 행: {frame.duplicated().sum():,}건')
    print(f'[{name}] {id_col} 중복 행: {frame[id_col].duplicated().sum():,}건')
    display(report)

quality_report('고객', customers, '고객번호')
quality_report('계좌', accounts, '계좌번호')
quality_report('거래', transactions, '거래번호')


[고객] 완전 중복 행: 0건
[고객] 고객번호 중복 행: 0건


,결측치 수,결측치 비율(%),고유값 수
고객번호,0,0.0,300
도시,0,0.0,297
신용점수,0,0.0,242
가입일,0,0.0,300


[계좌] 완전 중복 행: 0건
[계좌] 계좌번호 중복 행: 0건


,결측치 수,결측치 비율(%),고유값 수
계좌번호,0,0.0,455
고객번호,0,0.0,238
계좌유형,0,0.0,3
잔액_USD,0,0.0,455
개설일,0,0.0,455


[거래] 완전 중복 행: 5건
[거래] 거래번호 중복 행: 5건


,결측치 수,결측치 비율(%),고유값 수
거래번호,0,0.0,300
계좌번호,0,0.0,224
거래금액_USD,8,2.6,292
거래일시,0,0.0,300


## 3. 고객 1 ─ N 계좌 1 ─ N 거래 관계 검증

각 계좌가 정확히 한 고객에게 연결되는지, 계좌와 거래의 외래키가 원본에 존재하는지 확인합니다. 또한 고객별 계좌 수와 계좌별 거래 수 분포를 출력합니다.

In [4]:
# 같은 계좌가 서로 다른 고객번호에 연결되면 1:N 구조가 깨진 것입니다.
account_customer_conflict = (
    accounts.groupby('계좌번호')['고객번호'].nunique().gt(1).sum()
)

# 외래키가 원본 테이블에 없는 경우를 확인합니다.
orphan_accounts = accounts.loc[~accounts['고객번호'].isin(customers['고객번호'])]
orphan_transactions = transactions.loc[~transactions['계좌번호'].isin(accounts['계좌번호'])]

print(f'한 계좌에 여러 고객이 연결된 계좌 수: {account_customer_conflict:,}')
print(f'고객 원본에 없는 고객번호를 가진 계좌 수: {len(orphan_accounts):,}')
print(f'계좌 원본에 없는 계좌번호를 가진 거래 수: {len(orphan_transactions):,}')

print('\n고객별 계좌 수 분포')
display(accounts.groupby('고객번호')['계좌번호'].nunique().describe())
print('계좌별 거래 수 분포')
display(transactions.groupby('계좌번호')['거래번호'].nunique().describe())


한 계좌에 여러 고객이 연결된 계좌 수: 0
고객 원본에 없는 고객번호를 가진 계좌 수: 0
계좌 원본에 없는 계좌번호를 가진 거래 수: 0

고객별 계좌 수 분포


,계좌번호
count,238.000000
mean,1.911765
std,1.049708
min,1.000000
25%,1.000000
50%,2.000000
75%,2.000000
max,7.000000


계좌별 거래 수 분포


,거래번호
count,224.000000
mean,1.339286
std,0.643053
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,6.000000


## 4. 안전한 데이터 병합

거래를 기준으로 계좌와 고객을 연결합니다. `validate='many_to_one'`은 거래 여러 건이 계좌 또는 고객 한 건에만 연결되는지 강제하여, 예상하지 못한 행 폭증을 방지합니다.

In [5]:
# 거래번호 중복은 거래 건수·금액 집계를 왜곡하므로, 검증용 사본에서는 하나만 남깁니다.
transactions_check = transactions.drop_duplicates(subset='거래번호').copy()

df_merged = transactions_check.merge(
    accounts[['계좌번호', '고객번호', '개설일']],
    on='계좌번호', how='inner', validate='many_to_one'
).merge(
    customers[['고객번호', '가입일']],
    on='고객번호', how='inner', validate='many_to_one'
)

print(f'중복 제거 전 거래 행 수: {len(transactions):,}')
print(f'중복 제거 후 거래 행 수: {len(transactions_check):,}')
print(f'병합 후 검증 대상 행 수: {len(df_merged):,}')


중복 제거 전 거래 행 수: 305
중복 제거 후 거래 행 수: 300
병합 후 검증 대상 행 수: 300


## 5. 날짜 무결성 검증

날짜를 변환한 뒤 두 규칙을 **독립적으로** 검사합니다. 한 거래가 두 규칙을 모두 위반할 수 있으므로, `if/elif`로 하나만 분류하지 않습니다.

- 규칙 A: 가입일 ≤ 개설일
- 규칙 B: 개설일 ≤ 거래일시

In [7]:
date_cols = ['가입일', '개설일', '거래일시']
for col in date_cols:
    df_merged[col] = pd.to_datetime(df_merged[col], errors='coerce')

date_missing_rows = df_merged[date_cols].isna().any(axis=1).sum()
df_merged['오류_가입전개설'] = df_merged['개설일'] < df_merged['가입일']
df_merged['오류_개설전거래'] = df_merged['거래일시'] < df_merged['개설일']
df_merged['날짜순서_정상'] = (
    df_merged[date_cols].notna().all(axis=1)
    & ~df_merged['오류_가입전개설']
    & ~df_merged['오류_개설전거래']
)

# 가입일-개설일은 계좌 단위, 개설일-거래일시는 거래 단위로 요약합니다.
account_dates = df_merged.drop_duplicates(subset='계좌번호').copy()
summary = pd.DataFrame({
    '검증 항목': ['날짜 결측', '가입일 이전 계좌 개설', '개설일 이전 거래', '두 규칙을 모두 충족한 거래'],
    '단위': ['거래 행', '고유 계좌', '고유 거래', '고유 거래'],
    '건수': [
        int(date_missing_rows),
        int(account_dates['오류_가입전개설'].sum()),
        int(df_merged['오류_개설전거래'].sum()),
        int(df_merged['날짜순서_정상'].sum())
    ]
})
summary['비율(%)'] = (summary['건수'] / [len(df_merged), len(account_dates), len(df_merged), len(df_merged)] * 100).round(1)
display(summary)

print('오류 사례:')
display(df_merged.loc[~df_merged['날짜순서_정상'],
    ['거래번호', '고객번호', '계좌번호', '가입일', '개설일', '거래일시',
     '오류_가입전개설', '오류_개설전거래']].head(10))


,검증 항목,단위,건수,비율(%)
0,날짜 결측,거래 행,0,0.0
1,가입일 이전 계좌 개설,고유 계좌,96,42.9
2,개설일 이전 거래,고유 거래,289,96.3
3,두 규칙을 모두 충족한 거래,고유 거래,0,0.0


오류 사례:


,거래번호,고객번호,계좌번호,가입일,개설일,거래일시,오류_가입전개설,오류_개설전거래
0,TXNEENMT4D7JLSTY4,CUS023QWAFS5VB1,ACCH1YWWW9TT0Q1,2025-10-15 07:39:21,2021-08-04 04:22:29,2019-01-01 20:00:47,True,True
1,TXN2A0Y7NP74BA27O,CUS06ATN1VYTMIM,ACC9L0YFOK3RHVE,2021-08-03 09:12:32,2025-06-28 00:10:50,2019-01-02 07:34:07,False,True
2,TXNR4NBZYEE61YFUU,CUS02XUUA5FJTDT,ACCNOQA1TLUOOJK,2025-08-27 03:16:29,2024-08-09 16:45:07,2019-01-02 09:37:39,True,True
3,TXNJ86CGNF58EOWIF,CUS04XM9H1NY7C3,ACCCZ9XRBCLMYTL,2020-08-04 21:08:19,2024-12-18 18:31:52,2019-01-02 21:59:10,False,True
4,TXN88LF0AA9MXHU1W,CUS057W2O5OLDV9,ACC0UX0KXA1DMVO,2022-10-27 04:46:17,2023-12-17 20:23:21,2019-01-03 06:53:21,False,True
5,TXNQ4348QJ47Z50KF,CUS06OWK6PSGQ99,ACCABIJVUXPJYDV,2024-05-03 01:22:07,2021-02-25 09:40:06,2019-01-04 08:20:42,True,True
6,TXNEA7VNL79QEHXV6,CUS05PXT6RIDLOF,ACC9D3A53AABPVU,2019-01-01 06:53:09,2020-06-22 19:21:54,2019-01-04 15:52:03,False,True
7,TXN17QQT7Z4S8YUSK,CUS03WXTJ3ISNIJ,ACCO6VKLH0VOHNR,2020-02-21 19:10:23,2022-03-23 19:51:18,2019-01-05 03:11:50,False,True
8,TXN2LZ3A5CI8UQ2NG,CUS04RTM317B1O2,ACCR0XUXB77CH7K,2025-01-01 15:40:27,2022-03-10 20:06:10,2019-01-05 18:22:52,True,True
9,TXNOYN8SQ6WXEGU98,CUS06WPX2CYOS0F,ACCYSJXFIJJMMS4,2020-03-19 14:24:30,2024-12-06 01:42:16,2019-01-05 22:07:29,False,True


## 6. 보조 진단: 계좌별 오류 상태와 개설연도 분포

기존 노트북의 계좌별 분류와 개설연도 분포는 오류의 규모와 원인을 설명하는 데 유용하므로 유지합니다. 단, 이는 금융 분석이 아니라 데이터 오류를 설명하기 위한 진단입니다.

In [9]:
# 계좌별로 정상 거래가 하나라도 있는지, 모든 거래가 개설일 이전인지 분류합니다.
account_summary = df_merged.groupby('계좌번호').agg(
    총거래건수=('거래번호', 'nunique'),
    개설일이전거래건수=('오류_개설전거래', 'sum')
).reset_index()
account_summary['정상거래건수'] = (
    account_summary['총거래건수'] - account_summary['개설일이전거래건수']
)

def classify_account(row):
    if row['정상거래건수'] == 0:
        return '모든 거래가 개설일 이전'
    if row['개설일이전거래건수'] == 0:
        return '모든 거래가 개설일 이후'
    return '정상·비정상 거래 혼재'

account_summary['계좌상태'] = account_summary.apply(classify_account, axis=1)
display(account_summary['계좌상태'].value_counts().rename_axis('계좌 상태').reset_index(name='계좌 수'))

# 거래일 범위와 계좌 개설연도 분포를 함께 제시하면 시간 순서 오류를 설명하기 쉽습니다.
account_dates['개설연도'] = account_dates['개설일'].dt.year
print('거래일 범위:', df_merged['거래일시'].min(), '~', df_merged['거래일시'].max())
display(account_dates['개설연도'].value_counts().sort_index().rename_axis('개설연도').reset_index(name='계좌 수'))


,계좌 상태,계좌 수
0,모든 거래가 개설일 이전,218
1,모든 거래가 개설일 이후,5
2,정상·비정상 거래 혼재,1


거래일 범위: 2019-01-01 20:00:47 ~ 2019-05-17 14:47:20


,개설연도,계좌 수
0,2019,26
1,2020,34
2,2021,23
3,2022,27
4,2023,36
5,2024,45
6,2025,33


## 7. 결론: GIGO(Garbage In, Garbage Out)

날짜 무결성 위반 비율이 높다면 이 데이터로부터 거래 시점, 가입 후 경과 기간, 계좌 개설 후 거래량, 고객 생애주기 같은 시간 기반 금융 인사이트를 도출할 수 없습니다. 오류를 임의로 수정하면 새로운 가정을 넣는 것이므로 원본 데이터만으로는 검증할 수 없습니다.

따라서 위 검증 결과에서 핵심 오류가 대량으로 확인되면, **이 데이터는 분석용으로 부적합하다고 결론 내리고 본격적인 금융 분석을 진행하지 않습니다.** 신뢰 가능한 원본 데이터 또는 날짜 정의에 대한 제공자의 확인이 필요합니다.